In [1]:
import pandas as pd
import numpy as np

In [122]:
df_hh=pd.read_csv('df_hh.csv')
print(len(df_hh))
df_hh.head()

13551


,title,hh_link,address,company,short_description,experience,salary
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки"
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN


# Удаляем лишний столбец

Ссылка на вакансию нам не нужна - удалим этот столбец.

In [106]:
df_hh=df_hh.drop(columns='hh_link')

# Обогащаем датасет hh

## Добавляем признаки Регион и Страна

По городам присоединим регион и страну, воспользовавшись справочником из АПИ суперджоба.

In [107]:
print(df_hh['address'].nunique())
df_hh['address'].unique()

1549


<StringArray>
[                                       'Москва',
                         'Москва, р-н Басманный',
                      'Москва, р-н Дорогомилово',
                       'Москва, р-н Даниловский',
                    'Москва, р-н Красносельский',
                     'Москва, р-н Замоскворечье',
                       'Москва, р-н Пресненский',
                         'Москва, р-н Хамовники',
                       'Зеленоград, р-н Савёлки',
                             'деревня Румянцево',
 ...
                       'Калининградская область',
           'Витебск, улица Петруся Бровки, 50к7',
           'Махачкала, Хаджалмахинская улица, 1',
                 'Дмитров, Московская улица, 29',
            'Бишкек, улица Михаила Фрунзе, 533А',
 'Набережные Челны, Старосармановская улица, 29',
   'Ташкент, Яшнабадский район, улица Пунган, 7',
                  'Брест, проспект Машерова, 17',
                 'Подольск, Фабричный проезд, 2',
                    'Владивосто

Видим, что в столбце address не только город, но и р-н/улица. Оставим только город. Предварительно проверим пропуски.

In [108]:
df_hh[df_hh['address'].isna()]

,title,address,company,short_description,experience,salary
1752,Junior PHP разработчик,NaN,NaN,NaN,NaN,NaN


Один пропуск (пустая вакансия). Дропним эту строку.

In [109]:
df_hh=df_hh.drop(1752)
df_hh['address'].isna().sum()

np.int64(0)

In [110]:
df_hh['address']=df_hh['address'].apply(lambda x: x.split(',')[0])
df_hh['address'].unique()

<StringArray>
[                             'Москва',                          'Зеленоград',
                   'деревня Румянцево',                            'Тольятти',
     'посёлок городского типа Родники',                               'Химки',
                          'Никольское',                         'Красногорск',
                     'Санкт-Петербург',                           'г. Москва',
 ...
    'Черняховский муниципальный округ',                              'Латвия',
                  'посёлок Ольховатка',                             'Испания',
                           'Гвардейск',               'Бугрино (Ненецкий АО)',
 'посёлок городского типа Октябрьский',                            'Лимассол',
                  'Надеждинский район',             'Калининградская область']
Length: 558, dtype: str

In [111]:
df_hh[df_hh['address'].str.contains(r'\.')]

,title,address,company,short_description,experience,salary
191,Инженер-разработчик,г. Москва,ООО Сенсорика-М,датчики скорости и длины ИСД. - микрометры. Оп...,Опыт 1-3 года,NaN
197,Разработчик C++,г. Москва,ООО Сенсорика-М,Знание языков C / C++. Опыт написания достаточ...,Опыт 3-6 лет,NaN
11851,Инженер-разработчик,г. Москва,ООО Сенсорика-М,датчики скорости и длины ИСД. - микрометры. Оп...,Опыт 1-3 года,NaN
11863,Разработчик C++,г. Москва,ООО Сенсорика-М,Знание языков C / C++. Опыт написания достаточ...,Опыт 3-6 лет,NaN
12423,Продакт-менеджер (направление Посуда),Москва 37.539087,ООО Гауф Рус,Обязательно знание заводов по тематике и работ...,Опыт 3-6 лет,NaN
12535,"Продакт-менеджер по МБТ (кофемашины, термопоты...",Москва 37.539087,ООО Гауф Рус,Опыт запуска продуктов с китайскими заводами. ...,Опыт 3-6 лет,NaN


In [112]:
df_hh['address'] = df_hh['address'].replace({
    'г. Москва':'Москва',
    'Москва 37.539087':'Москва'
})
df_hh['address'].unique()

<StringArray>
[                             'Москва',                          'Зеленоград',
                   'деревня Румянцево',                            'Тольятти',
     'посёлок городского типа Родники',                               'Химки',
                          'Никольское',                         'Красногорск',
                     'Санкт-Петербург',                            'Нью-Йорк',
 ...
    'Черняховский муниципальный округ',                              'Латвия',
                  'посёлок Ольховатка',                             'Испания',
                           'Гвардейск',               'Бугрино (Ненецкий АО)',
 'посёлок городского типа Октябрьский',                            'Лимассол',
                  'Надеждинский район',             'Калининградская область']
Length: 556, dtype: str

In [113]:
towns_regions_countries=pd.read_csv('../towns_regions_contries.csv')
towns_regions_countries.head()

,title,region__title,country_title
0,Москва,Московская область,Россия
1,Санкт-Петербург,Ленинградская область,Россия
2,Новосибирск,Новосибирская область,Россия
3,Екатеринбург,Свердловская область,Россия
4,Нижний Новгород,Нижегородская область,Россия


In [114]:
df_hh_rich=df_hh.merge(towns_regions_countries, how='left', left_on='address',right_on='title')
df_hh_rich.head()

,title_x,address,company,short_description,experience,salary,title_y,region__title,country_title
0,Разработчик,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN,Москва,Московская область,Россия
1,Разработчик,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки",Москва,Московская область,Россия
2,Backend-разработчик,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN,Москва,Московская область,Россия
3,Разработчик ЦФТ,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN,Москва,Московская область,Россия
4,Разработчик C++,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN,Москва,Московская область,Россия


In [115]:
df_hh_rich = df_hh_rich.drop(columns='title_y')


In [116]:
df_hh_rich[df_hh_rich['address'].isin(towns_regions_countries['country_title'])]

,title_x,address,company,short_description,experience,salary,region__title,country_title
2300,Графический дизайнер,Россия,Librico,Опыт работы графическим,Опыт 3-6 лет,"от 80 000 ₽ за месяц, на руки",NaN,NaN
8021,Manual junior+/middle QA engineer,Армения,Ayo Support Solutions,Знание теории тестирования и видов тестировани...,Опыт 1-3 года,NaN,NaN,NaN
8827,QA Engineer (gamedev),Армения,The Open Platform,"More than 2 years of experience in QA, includi...",Опыт 3-6 лет,NaN,NaN,NaN
8873,Junior QA Engineer (Manual),Армения,ТОО Playrix,"Быстрый рост в геймдизайн, продюсирование или ...",Опыт 1-3 года,NaN,NaN,NaN
8893,QA Engineer (Junior/Middle),Армения,Gaijin Games,Опыт тестирования игровых проектов. — Знание с...,Опыт 1-3 года,NaN,NaN,NaN
9210,QA Engineer (Junior/Middle),Армения,Gaijin Games,Опыт тестирования игровых проектов. — Знание с...,Опыт 1-3 года,NaN,NaN,NaN
12901,Product Operations,Латвия,ATAS Ltd,"Характер: готовность напоминать, переспрашиват...",Опыт 1-3 года,NaN,NaN,NaN
12982,Project manager,Латвия,ATAS Ltd,Опыт работы с ATAS или платформами объёмного а...,Опыт 1-3 года,NaN,NaN,NaN


In [117]:
df_hh_rich.loc[2300, 'country_title'] = 'Россия'
df_hh_rich.loc[[8021, 8827,8873,8893,9210], 'country_title'] = 'Армения'
df_hh_rich.loc[[12901, 12982], 'country_title'] = 'Латвия'

## Добавляем признак client_staff_count (кол-во сотрудников компании)

In [118]:
company_staff=pd.read_csv('../company_staff.csv')
company_staff.head()

,firm_name,client_staff_count
0,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,100 — 500
1,Филиал ФКУ Налог-Сервис ФНС России по ЦОД в г....,100 — 500
2,Филиал ФКУ «Налог-Сервис» ФНС России по ЦОД в ...,100 — 500
3,Социальный фонд России,более 5000
4,Duolingo,менее 50


In [121]:
df_hh_rich=df_hh_rich.merge(company_staff, how='left', left_on='company',right_on='firm_name')
df_hh_rich['client_staff_count'].isna().sum()

np.int64(13516)

пупупу.. присоединилось маловато (из 13551). ну ничего, у нас еще много других более полезных признаков.

Скачаем датасет и перейдем к соединению двух датасетов и анализу итогового.

In [123]:
df_hh_rich.to_csv('../../analysis/df_hh_rich.csv', index=False)